# 03 · Open data APIs

**Session 4 — Wrangle and Version** · about 35 minutes

**You will learn to**
- read a dataset's metadata and licence before using it
- query Socrata (Transparència Catalunya) and CKAN (Open Data BCN) APIs
- download all rows with pagination and let the server aggregate for you
- cache downloads as Parquet and keep API tokens out of your code

**Data:** Transparència Catalunya and Open Data BCN (live), with offline copies in `data/examples/`.

**Outline**
0. Before you start (no AI)
1. Metadata first
2. Socrata queries
3. Pagination
4. Caching
5. CKAN: Open Data BCN
6. Tokens and secrets
7. Your dataset

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

1. A dataset has 54,611 rows and the API returns at most 10,000 rows per request.
   How many requests do you need?
2. Why save the downloaded data to a file instead of calling the API every time you run the notebook?
3. Where should an API token live: in the notebook, in a `.env` file, or in a message to an AI assistant?

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import requests

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

In [ ]:
SOCRATA = "https://analisi.transparenciacatalunya.cat"
OFFLINE = os.environ.get("OFFLINE") == "1"   # set OFFLINE=1 to use the copies in data/examples/
print("Offline mode" if OFFLINE else "Online mode")

## 1. Metadata first

Every Socrata dataset has a page (`/d/<id>`) and a metadata endpoint (`/api/views/<id>.json`)
with its description, publisher, update date, licence and column descriptions.

In [ ]:
DATASET = "8idm-becu"   # electricity consumption by municipality and sector

if OFFLINE:
    meta = {"name": "Consum d'energia elèctrica per municipis i sectors de Catalunya",
            "attribution": "(offline)", "license": {"name": "See Terms of Use"}, "columns": []}
else:
    meta = requests.get(f"{SOCRATA}/api/views/{DATASET}.json", timeout=30).json()

print(meta["name"])
print("Publisher:", meta.get("attribution"), "| Licence:", meta.get("license", {}).get("name"))
pd.DataFrame([{"field": c["fieldName"], "type": c["dataTypeName"], "description": c.get("description", "")[:60]}
              for c in meta["columns"]])

## 2. Socrata queries (SoQL)

The API understands SQL-like parameters: `$select`, `$where`, `$group`, `$order`, `$limit`, `$offset`.

In [ ]:
params = {"$where": "any = 2023 AND municipi = 'GIRONA'", "$order": "consum_kwh DESC"}
if OFFLINE:
    girona = (pd.read_parquet(DATA / "examples/energy_consumption_municipalities.parquet")
              .query("any == 2023 and municipi == 'GIRONA'")
              .sort_values("consum_kwh", ascending=False))
else:
    girona = pd.read_csv(f"{SOCRATA}/resource/{DATASET}.csv?" + requests.compat.urlencode(params))
girona[["any", "municipi", "descripcio_sector", "consum_kwh"]]

**Let the server do the work.** The COVID-19 vaccination dataset has more than a million rows,
but you can ask the server for daily totals and download about 1,600 rows instead:

In [ ]:
params = {"$select": "data, sum(recompte) as dosis", "$group": "data", "$order": "data", "$limit": 5000}
if OFFLINE:
    daily = (pd.read_parquet(DATA / "examples/covid_vaccination_daily.parquet")
             .groupby("data", as_index=False)["dosis"].sum())
else:
    daily = pd.read_csv(f"{SOCRATA}/resource/irki-p3c7.csv?" + requests.compat.urlencode(params))
daily["data"] = pd.to_datetime(daily["data"])
print(len(daily), "days")
daily.tail()

## 3. Pagination

Without `$limit`, Socrata returns only the first 1,000 rows — a classic silent bug.
Download page by page until a page comes back short:

In [ ]:
def fetch_socrata(dataset_id, params=None, page_size=10_000, max_rows=None):
    """Download a Socrata dataset as a DataFrame, page by page ($limit / $offset)."""
    params = dict(params or {})
    frames, offset = [], 0
    while True:
        query = {**params, "$limit": page_size, "$offset": offset, "$order": params.get("$order", ":id")}
        r = requests.get(f"{SOCRATA}/resource/{dataset_id}.json", params=query, timeout=60)
        r.raise_for_status()
        page = pd.DataFrame(r.json())
        frames.append(page)
        offset += page_size
        if len(page) < page_size or (max_rows and offset >= max_rows):
            break
    return pd.concat(frames, ignore_index=True)

In [ ]:
if OFFLINE:
    energy = pd.read_parquet(DATA / "examples/energy_consumption_municipalities.parquet")
    n_server = len(energy)
else:
    energy = fetch_socrata(DATASET)
    n_server = int(requests.get(f"{SOCRATA}/resource/{DATASET}.json",
                                params={"$select": "count(*)"}, timeout=30).json()[0]["count"])
print(f"downloaded {len(energy):,} rows; the server has {n_server:,}")
assert len(energy) == n_server, "Rows are missing: check the pagination."

`sodapy` is a small client library for the same API (it adds the token handling for you):

```python
from sodapy import Socrata
client = Socrata("analisi.transparenciacatalunya.cat", app_token=None)
rows = client.get("8idm-becu", where="any = 2023", limit=5000)
```

## 4. Caching

Download once, save as Parquet, then work from the file. The notebook stays fast,
works offline, and your analysis does not change when the portal updates the data
(re-download on purpose, and say so in your README).

In [ ]:
CACHE = DATA / "cache"          # git-ignored
CACHE.mkdir(exist_ok=True)


def cached(name, fetch):
    """Return the cached Parquet file `name`; call fetch() and save it if it does not exist yet."""
    path = CACHE / f"{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    df = fetch()
    df.to_parquet(path, index=False)
    return df


cooperatives = cached("cooperatives",
                      lambda: pd.read_parquet(DATA / "examples/cooperatives.parquet") if OFFLINE
                      else fetch_socrata("euku-fzbx"))
cooperatives["year"] = pd.to_datetime(cooperatives["data_d_inscripci"]).dt.year
ax = cooperatives["year"].value_counts().sort_index().plot(marker="o", figsize=(8, 3))
ax.set_xlabel("Year of registration")
ax.set_ylabel("Cooperatives registered")
plt.show()

## 5. CKAN: Open Data BCN

Barcelona's portal uses CKAN: `datastore_search` with `resource_id`, `limit` and `offset`.
People involved in traffic accidents handled by the Guàrdia Urbana in 2024 (CC BY 4.0):

In [ ]:
CKAN = "https://opendata-ajuntament.barcelona.cat/data/api/action/datastore_search"
RESOURCE = "c78c2bf7-88a6-4db9-a1e9-429f0ce14e88"

if OFFLINE:
    accidents = pd.read_parquet(DATA / "examples/bcn_accidents_2024.parquet")
else:
    result = requests.get(CKAN, params={"resource_id": RESOURCE, "limit": 32000}, timeout=60).json()["result"]
    accidents = pd.DataFrame(result["records"])
    print("server total:", result["total"])
accidents[["Nom_districte", "Desc_Tipus_vehicle_implicat", "Descripcio_victimitzacio"]].head()

In [ ]:
top_vehicles = accidents["Desc_Tipus_vehicle_implicat"].value_counts().head(8).sort_values()
ax = top_vehicles.plot.barh(figsize=(7, 3.5), color="tab:blue")
ax.set_xlabel("People involved (2024)")
ax.set_ylabel(None)
plt.show()

## 6. Tokens and secrets

Some APIs need a token (Socrata works without one but is faster with an *app token*).

1. Put it in a file called `.env` in the repository root: `SOCRATA_APP_TOKEN=abc123`
2. `.env` is listed in `.gitignore`, so it never reaches GitHub.
3. Read it in code with `python-dotenv`. Never print it, never paste it in a notebook you push,
   and never paste it into a chat with an AI assistant.

In [ ]:
from dotenv import load_dotenv

load_dotenv()                                   # reads ../.env if it exists
token = os.environ.get("SOCRATA_APP_TOKEN")     # None if you have no token
headers = {"X-App-Token": token} if token else {}
print("token found" if token else "no token: fine for small downloads")

## 7. Your dataset

### TASK — load your data reproducibly

In your group repository, write a notebook that:

1. prints the dataset's name, publisher, licence and last update (metadata);
2. downloads **all** rows (pagination) or lets the server aggregate what you need;
3. checks the row count against the server (`$select=count(*)`);
4. caches the result as Parquet in a git-ignored folder;
5. reads any token from `.env`.

Commit it with a message that says why (e.g. *"Download school results via API and cache to Parquet"*).

**Next:** `04_git_and_github.ipynb`.